# ICBHI 2017 Respiratory Sound Classification — Ablation Study

This notebook reproduces the respiratory-sound binary classification framing from:
> **RespLLM: Unifying Audio and Text with Multimodal LLMs for Generalized Respiratory
> Health Prediction** — using the ICBHI 2017 Respiratory Sound Database.

It demonstrates:
1. Loading the ICBHI dataset with `ICBHIDataset`
2. Applying `RespiratoryAbnormalityPredictionICBHI`
3. **Ablation over three `label_mode` variants** — `any_abnormal`, `crackle_only`, `wheeze_only`
4. Training a minimal 1-D CNN for each mode and comparing class balance and test accuracy

**Running this notebook:**
- Set `DEMO = True` (default) to run on purely synthetic data — no download required.
- Set `DEMO = False` and `ROOT` to your `ICBHI_final_database/` path for real results.

**Dataset:** https://bhichallenge.med.auth.gr/ICBHI_2017_Challenge  
**Citation:** Rocha et al., *Physiological Measurement* 40.3 (2019): 035001.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — adjust before running
# ---------------------------------------------------------------------------
ROOT = "/data/ICBHI_final_database"   # used only when DEMO=False
CACHE_DIR = ".cache/icbhi_ablation"

DEMO = True           # True -> synthetic data, no download needed

RESAMPLE_RATE = 4000  # Hz — low enough to capture respiratory acoustics
TARGET_LENGTH = 2.0   # seconds — pad/trim each cycle to this fixed length
BATCH_SIZE = 4
EPOCHS = 3            # short for demo; use 20+ with real data

In [ ]:
import shutil
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from pyhealth.datasets import ICBHIDataset
from pyhealth.tasks import RespiratoryAbnormalityPredictionICBHI
from pyhealth.tasks.icbhi_respiratory_classification import _LABEL_NAMES_BY_MODE

## Demo Fixture

When `DEMO=True`, we synthesize a minimal ICBHI-shaped directory: 5 patients, 1 recording each,
2 cycles per recording covering all four crackle/wheeze annotation combinations.  
Audio is zero-filled (no real acoustic content) — training does not converge, but every code path
executes correctly so you can inspect the ablation structure.

In [ ]:
def _build_demo_root(root: Path) -> None:
    """Populate *root* with a minimal ICBHI-shaped synthetic dataset.

    5 patients, 1 recording each, 2 cycles per recording.  Covers all four
    (crackle, wheeze) combinations: (0,0), (1,0), (0,1), (1,1).
    3 patients in the train split, 2 in the test split.
    """
    import scipy.io.wavfile

    sample_rate = RESAMPLE_RATE
    audio = np.zeros(sample_rate * 4, dtype=np.int16)   # 4-second silent clip

    # (patient_id, rec, location, mode, equipment, split, [(start,end,crackle,wheeze)...])
    recordings = [
        ("101", "1b1", "Ar", "sc", "Meditron", "train",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 1, 0)]),   # normal, crackle
        ("102", "1b1", "Pl", "mc", "AKGC417L", "train",
         [(0.0, 1.5, 0, 1), (1.5, 3.0, 1, 1)]),   # wheeze, both
        ("103", "1b1", "Ar", "sc", "Meditron", "train",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 0, 1)]),   # normal, wheeze
        ("104", "1b1", "Ar", "sc", "Meditron", "test",
         [(0.0, 1.5, 1, 0), (1.5, 3.0, 1, 1)]),   # crackle, both
        ("105", "1b1", "Ar", "sc", "Meditron", "test",
         [(0.0, 1.5, 0, 0), (1.5, 3.0, 0, 1)]),   # normal, wheeze
    ]

    train_stems, test_stems = [], []
    for pid, rec, loc, mode, equip, split, anns in recordings:
        stem = f"{pid}_{rec}_{loc}_{mode}_{equip}"
        scipy.io.wavfile.write(str(root / f"{stem}.wav"), sample_rate, audio)
        ann_text = "\n".join(f"{s}\t{e}\t{c}\t{w}" for s, e, c, w in anns)
        (root / f"{stem}.txt").write_text(ann_text + "\n")
        (train_stems if split == "train" else test_stems).append(stem)

    (root / "ICBHI_challenge_diagnosis.txt").write_text(
        "101 Healthy\n102 URTI\n103 COPD\n104 URTI\n105 Healthy\n"
    )
    (root / "ICBHI_Challenge_demographic_information.txt").write_text(
        "101 45 M 22.5 NA NA\n102 5 F NA 18.2 108.0\n"
        "103 67 M 28.1 NA NA\n104 12 F NA 21.0 140.0\n105 55 M 24.3 NA NA\n"
    )
    split_dir = root / "ICBHI_challenge_train_and_test_txt"
    split_dir.mkdir()
    (split_dir / "ICBHI_challenge_train.txt").write_text("\n".join(train_stems) + "\n")
    (split_dir / "ICBHI_challenge_test.txt").write_text("\n".join(test_stems) + "\n")

## Model and Helpers

A lightweight 1-D CNN baseline: two conv layers + adaptive pooling → linear head.
Output is **2-class** (binary: normal vs. abnormal under the selected `label_mode`).

This is intentionally minimal — the goal is to show how class balance and task framing
change across ablation modes, not to achieve state-of-the-art accuracy.

In [ ]:
class RespiratoryConv1D(nn.Module):
    """Lightweight 1-D CNN for binary respiratory sound cycle classification."""

    def __init__(self, n_classes: int = 2, target_samples: int = 8000) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=8, padding=28),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=32, stride=4, padding=12),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(16),
        )
        self.classifier = nn.Linear(32 * 16, n_classes)

    def forward(self, signal: torch.Tensor) -> torch.Tensor:
        x = self.encoder(signal)        # (B, 32, 16)
        x = x.flatten(start_dim=1)     # (B, 512)
        return self.classifier(x)      # (B, n_classes)


def collate_fn(batch):
    signals = torch.stack([item["signal"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return signals, labels

## Ablation Function

Each `label_mode` variant defines what counts as "abnormal":

| `label_mode`    | Positive class definition                    | Negative / Positive names    |
|-----------------|----------------------------------------------|------------------------------|
| `any_abnormal`  | `has_crackles OR has_wheezes`                | normal / abnormal            |
| `crackle_only`  | `has_crackles`                               | no_crackle / crackle         |
| `wheeze_only`   | `has_wheezes`                                | no_wheeze / wheeze           |

Each call to `run_mode` trains a fresh model from scratch on the same dataset, so the only
variable is the label definition. This isolates how the choice of target affects class balance
and learnability.

In [ ]:
def run_mode(dataset: ICBHIDataset, label_mode: str) -> dict:
    """Train and evaluate a binary 1-D CNN for one label_mode variant.

    Args:
        dataset: Pre-loaded ICBHIDataset (shared across ablation modes).
        label_mode: One of 'any_abnormal', 'crackle_only', 'wheeze_only'.

    Returns:
        dict with keys: label_mode, n_train, n_test, train_pos_pct, test_acc.
    """
    neg_name, pos_name = _LABEL_NAMES_BY_MODE[label_mode]
    print(f"\n{'='*60}")
    print(f"  ABLATION: label_mode = {label_mode!r}")
    print(f"  Labels: 0={neg_name!r}  1={pos_name!r}")
    print(f"{'='*60}")

    task = RespiratoryAbnormalityPredictionICBHI(
        label_mode=label_mode,
        resample_rate=RESAMPLE_RATE,
        target_length=TARGET_LENGTH,
    )
    samples = dataset.set_task(task)

    # Split by the official ICBHI train/test annotation embedded in each sample.
    all_samples = list(samples)
    train_samples = [s for s in all_samples if s["split"] == "train"]
    test_samples  = [s for s in all_samples if s["split"] == "test"]

    n_train = len(train_samples)
    n_test  = len(test_samples)
    train_pos = sum(1 for s in train_samples if s["label"] == 1)
    train_pos_pct = train_pos / n_train if n_train else 0.0

    print(f"  Train: {n_train} cycles  "
          f"({train_pos}/{n_train} positive = {train_pos_pct:.1%})")
    print(f"  Test:  {n_test} cycles")

    if n_train == 0:
        print("  [SKIP] no training samples — returning NaN accuracy")
        return {"label_mode": label_mode, "n_train": 0, "n_test": n_test,
                "train_pos_pct": 0.0, "test_acc": float("nan")}

    target_samples_n = int(TARGET_LENGTH * RESAMPLE_RATE)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = RespiratoryConv1D(n_classes=2, target_samples=target_samples_n).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        train_samples, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=0,
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for signals, labels in train_loader:
            signals, labels = signals.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(signals)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
        train_acc = correct / total if total else 0.0
        print(f"  Epoch {epoch}/{EPOCHS} — "
              f"loss: {total_loss / total:.4f}  "
              f"train acc: {train_acc:.3f}")

    if test_samples:
        test_loader = DataLoader(
            test_samples, batch_size=BATCH_SIZE, shuffle=False,
            collate_fn=collate_fn, num_workers=0,
        )
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for signals, labels in test_loader:
                signals, labels = signals.to(device), labels.to(device)
                correct += (model(signals).argmax(1) == labels).sum().item()
                total += len(labels)
        test_acc = correct / total if total else 0.0
    else:
        test_acc = float("nan")

    print(f"  Test accuracy: {test_acc:.3f}")

    try:
        samples.close()
    except AttributeError:
        pass

    return {
        "label_mode": label_mode,
        "n_train": n_train,
        "n_test": n_test,
        "train_pos_pct": train_pos_pct,
        "test_acc": test_acc,
    }

## Load Dataset

In [ ]:
_tmpdir = None

if DEMO:
    _tmpdir = tempfile.mkdtemp(prefix="icbhi_demo_")
    _demo_root = Path(_tmpdir)
    print(f"Demo mode: building synthetic fixture at {_demo_root}")
    _build_demo_root(_demo_root)
    _root = str(_demo_root)
    _cache = str(_demo_root / ".cache")
else:
    _root = ROOT
    _cache = CACHE_DIR

print(f"Loading ICBHIDataset from: {_root}")
dataset = ICBHIDataset(root=_root, subset="both", cache_dir=_cache)
dataset.stats()

## Ablation: three `label_mode` variants

We train a fresh 1-D CNN for each mode using the official ICBHI train split and evaluate on the
official test split.  The shared dataset is loaded once; only the label function changes.

**Expected observations on real ICBHI data (920 recordings, 6898 cycles):**
- `any_abnormal`: ~57% positive — most balanced; broadest positive class
- `crackle_only`: ~40% positive — crackles dominate adventitious sounds in ICBHI
- `wheeze_only`: ~13% positive — rarest class; highest imbalance; hardest to learn

Models will generally achieve higher accuracy on `any_abnormal` due to better class balance.
The `wheeze_only` mode is the most challenging ablation.

*In demo mode, audio is zero-filled so results are essentially random — the point is to verify*
*that all three modes run correctly and produce different class distributions.*

In [ ]:
# Ablation modes in order of expected difficulty (easiest to hardest)
ABLATION_MODES = ["any_abnormal", "crackle_only", "wheeze_only"]

results = []
for mode in ABLATION_MODES:
    results.append(run_mode(dataset, mode))

In [ ]:
# ---------------------------------------------------------------------------
# Ablation summary table
# ---------------------------------------------------------------------------
print("\n" + "=" * 62)
print("  ABLATION SUMMARY")
print("=" * 62)
print(f"  {'Mode':<20} {'N train':>8} {'Train +%':>10} {'Test Acc':>10}")
print("  " + "-" * 54)
for r in results:
    acc_str = f"{r['test_acc']:.3f}" if r["test_acc"] == r["test_acc"] else "  N/A  "
    print(
        f"  {r['label_mode']:<20} {r['n_train']:>8} "
        f"{r['train_pos_pct']:>9.1%} {acc_str:>10}"
    )
print("=" * 62)
print("""
Interpretation:
  any_abnormal  — broadest positive class; best class balance on real data
  crackle_only  — crackles are dominant; moderate imbalance
  wheeze_only   — rarest sound; highest imbalance; hardest to learn

Demo mode: 10 synthetic train / 4 test cycles; results are meaningless.
Set DEMO=False with a real ICBHI download for interpretable results.
""")

In [ ]:
# Cleanup synthetic temp directory (no-op when DEMO=False)
if _tmpdir:
    shutil.rmtree(_tmpdir, ignore_errors=True)
    print(f"Cleaned up demo temp dir: {_tmpdir}")